In [ ]:
import os
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reproduce.py").is_file())
os.chdir(ROOT)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROJECT_PATH = Path("data/genomics")

In [ ]:
meta = pd.read_csv(PROJECT_PATH / "reference/metadata_complete.csv")
meta['population'] = meta['Strain'] + '_' + meta['Culture'].astype(str).str.zfill(2)
meta

In [ ]:
# meta.query('population=="PLAC_01"')

In [ ]:
dfs = []

for i, row in meta.iterrows():
    df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
    df["Strain"] = row['Strain']
    df["Culture"] = row['Culture']
    df["Day"] = int(row['Day'])
    dfs.append(df)

df = pd.concat(dfs)
df

In [ ]:
select_lineage = "ATEC-C"
select_lineage_2 = "ATEC-C-R"
pops = meta.query(f'Strain in ["{select_lineage}", "{select_lineage_2}"]')['population'].unique()
timepoints = sorted(meta.query(f'Strain in ["{select_lineage}", "{select_lineage_2}"]')['Day'].unique())
print(pops)
print(timepoints)

In [ ]:
meta['Strain'].unique()

In [ ]:
meta.query(f'population=="{select_lineage}_01"')

In [ ]:
def traceAlleleFreq(pop, min_freq=0.33):

    D = []
    sorted_meta = meta.query(f'population=="{pop}"').sort_values(by='Day')
    for i, row in sorted_meta.iterrows():
        df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
        df["abs_position"] = df['seq_id'] + ' ' + df['position'].astype(str)
        D.append(df)

    # Remove mutations detected in the wild-type background
    # 1. Select background mutations with strong signal
    wt_background=D[0].loc[D[0].frequency>0.01, 'abs_position']
    #print(f"WT background mutations: {wt_background.values}")
    D_=[]

    #print(f"# of mutations\t# of (freq>{min_freq}):")
    for i in range(len(D)):
        # 2. Filter out mutations by position on the chromosome
        df=D[i][~D[i]['abs_position'].isin(wt_background)]
        D_.append(df)
        #print(f"{df.shape[0]}\t{sum(df['frequency']>min_freq)}")
    
    tracked_positions=[]
    for i in range(len(D_)):
        d=D_[i]
        tracked_positions.extend(list(d.loc[ (d['frequency']>min_freq), 'abs_position' ])) #& ~d['mutation_category'].isin(['mobile_element_insertion','large_deletion']), 'position' ] ))
    tracked_positions=set(tracked_positions)
    print(f"Population {pop} # of tracked mutations: {len(tracked_positions)}")

    ## 3. Trace the frequency of those mutations
    T=[]
    for pos in tracked_positions:

        freq=[]
        for i in range(len(D_)):
            d = D_[i]
            ind = d['abs_position']==pos
            
            if sum(ind)<1:
                freq.append(0)
            else:
                freq.append( (d.loc[ind , 'frequency'].values)[0] )
                gene_name = d.loc[ind, 'gene_name'].values[0]
                gene_product = d.loc[ind, 'gene_product'].values[0]
                aa_ref_seq = d.loc[ind, 'aa_ref_seq'].values[0]
                aa_new_seq = d.loc[ind, 'aa_new_seq'].values[0]
                aa_ref_seq = d.loc[ind, 'aa_ref_seq'].values[0]
                new_seq = d.loc[ind, 'new_seq'].values[0]
                codon_ref_seq = d.loc[ind, 'codon_ref_seq'].values[0]
                aa_pos = d.loc[ind, 'aa_position'].values[0]
                gene_pos = d.loc[ind, 'gene_position'].values[0]
                mut_cat = d.loc[ind, 'mutation_category'].values[0]

        T.append({'abs_position': pos, 'freq': np.round(freq,2), 
                'gene_name':gene_name, 'gene_product':gene_product,
                'aa_ref_seq':aa_ref_seq, 'aa_new_seq':aa_new_seq,
                'aa_pos': aa_pos, 'gene_pos':gene_pos, 'mut_cat': mut_cat,
                'new_seq': new_seq, 'codon_ref_seq': codon_ref_seq})

    T=pd.DataFrame(T)
    T.fillna({'aa_ref_seq': '',
              'aa_new_seq': '',
              'aa_pos': ''},inplace=True)
    
    nsi = T['aa_pos']==''
    T.loc[nsi,'label'] = T.loc[nsi,'gene_name'] + ' ' + T.loc[nsi, 'mut_cat']
    T.loc[~nsi, 'label'] = T.loc[~nsi,'gene_name'] + ' ' + T.loc[~nsi,'aa_ref_seq'] + T.loc[~nsi, 'aa_pos'].astype(str).str.replace(r'\.0','',regex=True) + T.loc[~nsi,'aa_new_seq']
        # T.loc[nsi,'gene_pos'] + ' '
    T.sort_values(by=['mut_cat','gene_name'],ascending=False,inplace=True)
    T.reset_index(inplace=True,drop=True)
    
    return T

In [ ]:
af=[]
for pop in pops:
    # print(traceAlleleFreq(pop, min_freq=0.2).shape)
    af.append(traceAlleleFreq(pop, min_freq=0.1))
    af[-1].to_csv(f"data/genomics/data/processed/traced_alleles/{select_lineage}/{pop}.csv", index=False)

In [ ]:
print(af[0].columns)

In [ ]:
af[2]

In [ ]:
clt = 7

selB_rows = af[clt-1].query('gene_product=="Putative multidrug resistance outer membrane protein MdtQ/tRNA-dihydrouridine(16) synthase"')

# Display the filtered rows
selB_rows

In [ ]:
af[3].loc[
    (af[3]['gene_name'] == 'phoQ') & (af[3]['gene_pos'] == 'coding (1265-1268/1461 nt)'),
    'label'
] = 'phoQ IS2 insertion'
#
af[3].loc[
    (af[3]['gene_name'] == 'phoQ') & (af[3]['gene_pos'] == 'coding (136-140/1461 nt)'),
    'label'
] = 'phoQ IS5 insertion'

In [ ]:
pd.set_option("display.max_rows", 100)
af[0].head(100)

In [ ]:
all = []
for ix, df in enumerate(af):
    dff = df.copy()
    dff['Pop'] = ix+1 
    all.append(dff)
combined_df=pd.concat(all)


combined_df['last_freq'] = combined_df['freq'].apply(lambda x: x[-1])
combined_df

In [ ]:
# Generate a list of all unique values from the 'gene_product' column
gene_product_names = combined_df['gene_product'].unique()
gene_product_names

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors

# Create a new colormap that interpolates between white and the 'Greys' colormap
standard_map = plt.cm.get_cmap('Greys')
new_colors = standard_map(np.linspace(0, 1, 256))
new_colors[0] = np.array([1, 1, 1, 1])  # Replace the first color (for 0 values) with white
custom_map = mcolors.ListedColormap(new_colors)

# Generate a list of all unique values from the 'gene_product' column
####
    #    'H repeat-associated protein, 
    #    'YhhI family',
    #    'tRNA-Gln-TTG',
    #    'Aquaporin Z/Lysine exporter LysO',
    #    'UPF0149 exported protein YgfB', 
    #    'Predicted Fic antitoxin YfhG/Peptidyl-prolyl cis-trans isomerase PpiA precursor (EC 5.2.1.8)',
    #    'Putative glutamine amidotransferase YafJ/D-sedoheptulose 7-phosphate isomerase (EC 5.3.1.28)',
    #    '6-phosphogluconate dehydrogenase, decarboxylating (EC 1.1.1.44)/–',
    #    'Putative multidrug resistance outer membrane protein MdtQ/tRNA-dihydrouridine(16) synthase',
###
gene_product_names =np.array(['Toxin HigB / Protein kinase domain of HipA',
    #    'hypothetical protein',
    #    'Uncharacterized radical SAM protein YjjW/hypothetical protein',
    #    '[SEED:fig_6666666.487694.peg.2908]', 
       'FhuE receptor precursor',
       'Methionyl-tRNA synthetase (EC 6.1.1.10)',
    #    'hypothetical protein/Methylisocitrate lyase (EC 4.1.3.30)', intergenic
       'Phenylalanyl-tRNA synthetase alpha chain (EC 6.1.1.20)',
    #    'tRNA-Ala-TGC', #noncoding
    #    'IS1 protein InsB', 
       'Phenylalanyl-tRNA synthetase beta chain (EC 6.1.1.20)', #evi
       'Cysteinyl-tRNA synthetase (EC 6.1.1.16)',
       'Adenylate kinase (EC 2.7.4.3)',
       'Cell division protein FtsI [Peptidoglycan synthetase] (EC 2.4.1.129)',
       'HTH-type transcriptional repressor ComR',
       'Multiple antibiotic resistance protein MarR',
       'Copper sensory histidine kinase CpxA',
       'Multidrug efflux system AcrAB-TolC, inner-membrane proton/drug antiporter AcrB (RND type)',
       'ATP synthase beta chain (EC 3.6.3.14)',
       'Outer membrane porin OmpC',
       'ATP synthase epsilon chain (EC 3.6.3.14)',
       'Multidrug efflux system AcrAB-TolC, membrane fusion component AcrA/Transcriptional regulator of acrAB operon, AcrR',
       'Undecaprenyl-phosphate galactosephosphotransferase (EC 2.7.8.6)', 'H repeat-associated protein', 
       'YhhI family',
       'tRNA-Gln-TTG', #noncoding
       'Aquaporin Z/Lysine exporter LysO',
       'UPF0149 exported protein YgfB', 
    #    'Predicted Fic antitoxin YfhG/Peptidyl-prolyl cis-trans isomerase PpiA precursor (EC 5.2.1.8)', #after protein 
    #    'Putative glutamine amidotransferase YafJ/D-sedoheptulose 7-phosphate isomerase (EC 5.3.1.28)', #intergeneic and looks like a barcode
       '6-phosphogluconate dehydrogenase, decarboxylating (EC 1.1.1.44)/–',
       'Putative multidrug resistance outer membrane protein MdtQ/tRNA-dihydrouridine(16) synthase',
       
       ],
      dtype=object)
# Create a dictionary of alternate labels
alternate_labels_dict = {
    'Outer membrane porin OmpC': 'OmpC W93*', #confirmed
    'Multiple antibiotic resistance protein MarR': 'MarR R77C', #confirmed
    'Copper sensory histidine kinase CpxA': 'CpxA L30R', #confirmed
    'HTH-type transcriptional repressor ComR': 'ComR P16Q', #confirmed
    'Cell division protein FtsI [Peptidoglycan synthetase] (EC 2.4.1.129)': 'FtsI V313M', #confirmed
    'Multidrug efflux system AcrAB-TolC, membrane fusion component AcrA/Transcriptional regulator of acrAB operon, AcrR': r'$\it{acrA}$ intergenic snp', #cofnirmed r'$\it{acrA}$
    'FhuE receptor precursor': 'FhuE E619D', #confirmed
    # 'tRNA-Ala-TGC': 'alaW', #found trna for alanine
    'Multidrug efflux system AcrAB-TolC, inner-membrane proton/drug antiporter AcrB (RND type)': 'AcrB V127G', #confirmed
    'ATP synthase beta chain (EC 3.6.3.14)': 'AtpD S342R',#cefR
    'ATP synthase epsilon chain (EC 3.6.3.14)': 'AtpC Q25*',#cefR
    'Undecaprenyl-phosphate galactosephosphotransferase (EC 2.7.8.6)': 'WbaP P32fs', #cefR
    'Toxin HigB / Protein kinase domain of HipA': 'HipA G118D', #pbcef-1
    'Methionyl-tRNA synthetase (EC 6.1.1.10)': 'MetG V583M', #pbcef-2
    'Phenylalanyl-tRNA synthetase beta chain (EC 6.1.1.20)': 'PheT indel',#pbcef-4
    'Phenylalanyl-tRNA synthetase alpha chain (EC 6.1.1.20)': 'PheS L24F', #pbcef-3
    'UPF0149 exported protein YgfB': 'YgfB P184Q', #pbcef-3
    'tRNA-Gln-TTG': 'GlnW snp',
    'Cysteinyl-tRNA synthetase (EC 6.1.1.16)': 'CysS V27G',#pbcef-5
    'Adenylate kinase (EC 2.7.4.3)': 'Adk D147Y', #pbcef-6
    '6-phosphogluconate dehydrogenase, decarboxylating (EC 1.1.1.44)/–': r'$\it{gnd}$ snp intergenic' ,#pbcef-6 r'$\it{gnd}$
    'Aquaporin Z/Lysine exporter LysO': r'$\it{aqpZ/lysO}$ snp intergenic', #pbcef-7 
    'Putative multidrug resistance outer membrane protein MdtQ/tRNA-dihydrouridine(16) synthase' :r'$\it{mdtQ/dusC}$ snp intergenic', 



    # 'hypothetical protein/Methylisocitrate lyase (EC 4.1.3.30)': 'prpB', #pbcef-2
    # 'IS1 protein InsB': 'IS1 InsB', #pbcef-4 #convoluted  #pbcef-4
}   


# Map the gene_product values in the DataFrame to the alternate labels
combined_df['alternate_label'] = combined_df['gene_product'].map(alternate_labels_dict)

# Define the y-axis labels (Pop to strain mapping with "CEF" in non-italic superscript)
pop_labels = {
    1: r'Pb$^{\mathrm{CEF}}$-1', 2: r'Pb$^{\mathrm{CEF}}$-2', 3: r'Pb$^{\mathrm{CEF}}$-3',
    4: r'Pb$^{\mathrm{CEF}}$-4', 5: r'Pb$^{\mathrm{CEF}}$-5', 6: r'Pb$^{\mathrm{CEF}}$-6',
    7: r'Pb$^{\mathrm{CEF}}$-R-1'
}

# Map the gene_product values in the DataFrame to alternate labels for labeling
combined_df['plot_label'] = combined_df['gene_product'].map(alternate_labels_dict).fillna(combined_df['gene_product'])

# Filter the DataFrame for specified gene product names
filtered_df = combined_df[combined_df['gene_product'].isin(gene_product_names)]

# Pivot the DataFrame to have 'plot_label' as rows, 'Pop' as columns, and 'last_freq' as values
heatmap_data = filtered_df.pivot_table(index='plot_label', columns='Pop', values='last_freq', fill_value=0)

# Convert the DataFrame to a numpy array for sorting
freq_mat = heatmap_data.values

# Sort the rows by the values in Pop 7 in descending order
sort_index = np.argsort(heatmap_data[7])[::-1]  # Replace 7 with the column name for Pop 7 if needed
freq_mat = freq_mat[sort_index]  # Reorder frequency matrix based on sorted index
sorted_labels = heatmap_data.index[sort_index]  # Get the sorted labels

# Timepoints (populations) for y-axis
pop_labels = {
    1: r'Pb$^{\mathrm{CEF}}$-1', 2: r'Pb$^{\mathrm{CEF}}$-2', 3: r'Pb$^{\mathrm{CEF}}$-3',
    4: r'Pb$^{\mathrm{CEF}}$-4', 5: r'Pb$^{\mathrm{CEF}}$-5', 6: r'Pb$^{\mathrm{CEF}}$-6',
    7: r'Pb$^{\mathrm{CEF}}$-R-1'
}
timepoints = [pop_labels[pop] for pop in heatmap_data.columns]

# Create annotation matrix to hide values where they are 0
annot_matrix = np.where(freq_mat.T == 0, '', freq_mat.T.round(1).astype(str))

# Plotting the heatmap
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(freq_mat.T, yticklabels=timepoints, xticklabels=sorted_labels, cmap=custom_map, vmin=0, vmax=1, ax=ax,
            annot=annot_matrix, fmt="", annot_kws={'size': 12}, cbar=False, linewidths=0.5, linecolor='lightgrey')

# Customize plot labels
ax.set_ylabel('', fontsize=14)
ax.set_xlabel('', fontsize=14)
ax.tick_params(axis='both', which='major', labelsize=12, length=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=12, rotation_mode='anchor')
ax.set_yticklabels(ax.get_yticklabels(), fontsize=12)

# Adjust spines
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)
    spine.set_color("black")

# Display the plot
plt.tight_layout()
plt.show()

# Save the figure (replace PROJECT_PATH with your project path)
fig.savefig(PROJECT_PATH / "figures/final/pbC_full.png", dpi=300, bbox_inches='tight')
